# ECG Signal Explorer

In [1]:
import copy
from kymira_data_in import read_ecg_trial
# Pre-processing functions
from kymira_ecg.preprocessing import butterworth_bandpass_filtering, notch_filtering, wavelet_denoising
from kymira_ecg.visualisation import plot_ecg_bokeh
import ipywidgets

## Work on:

In [2]:
channels = ["ECG_1","ECG_2","A_X","A_Y","A_Z","G_X","G_Y","G_Z","M_X","M_Y","M_Z","M_rH"]

In [12]:
Location = "/home/protopi/RaspberryPi_Dev/Test_Brain-Beta-v1-1_13-55_2022-05-19_v0.csv"
kymira_ecg_data = read_ecg_trial(Location)
print(kymira_ecg_data)

          ECG_1     ECG_2       A_X       A_Y       A_Z       G_X       G_Y  \
0     -0.179109 -0.155831  0.016113 -0.004883  1.015137 -0.002441 -0.003418   
1     -0.174217 -0.158645  0.016113 -0.004883  1.015137 -0.002441 -0.003418   
2     -0.167449 -0.161663  0.016113 -0.004883  1.015137 -0.002441 -0.003418   
3     -0.164558 -0.161526  0.016113 -0.004883  1.015137 -0.002441 -0.003418   
4     -0.163341 -0.161364  0.016113 -0.004883  1.015137 -0.002441 -0.003418   
...         ...       ...       ...       ...       ...       ...       ...   
18077 -0.162548 -0.162549  0.069336 -0.000488  1.015625 -0.002930  0.000977   
18078 -0.162547 -0.162547  0.069336 -0.000488  1.015625 -0.002930  0.000977   
18079 -0.162549 -0.162547  0.069336 -0.000488  1.015625 -0.002930  0.000977   
18080 -0.162546 -0.162545  0.066406 -0.000488  1.011230 -0.000977 -0.001953   
18081 -0.162548 -0.162545  0.066406 -0.000488  1.011230 -0.000977 -0.001953   

            G_Z       M_X       M_Y       M_Z      

## Time domain plot:

In [13]:
plot_ecg_bokeh(kymira_ecg_data,channels=channels)

Loading BokehJS ...

## Pre-processing

In [5]:
def temp_preprocess(apply_notch, apply_detrend, wt_denoise_level):
    """
    Applies the pre-processing.
    
    Notes:
        * This is a TEMPORARY function (the pre-processing pipeline is formalised).
    """
    pre_processed = copy.deepcopy(kymira_ecg_data)
    
    for a_channel in channels:
        stage_1 = notch_filtering(pre_processed[f"ecg{a_channel}"]) if apply_notch else pre_processed[f"ecg{a_channel}"]
        stage_2 = butterworth_bandpass_filtering(stage_1, 0.5, 40, 4, 500) if apply_detrend else stage_1
        stage_3 = wavelet_denoising(stage_2, denoise_level=1.0-wt_denoise_level)
        pre_processed[f"ecg{a_channel}"] = stage_3
        
    plot_ecg_bokeh(pre_processed,channels=channels)
    
    

ipywidgets.interact(temp_preprocess, 
                    apply_notch = ipywidgets.Checkbox(value=False, description="50 Hz notch", indent=False), 
                    apply_detrend = ipywidgets.Checkbox(value=False, description="0.5Hz - 40Hz Bandpass filter", indent=False),
                    wt_denoise_level = ipywidgets.FloatSlider(value=0, min=0.0, max=1.0, step=0.01, continuous_update=False, description="Wavelet denoise level"))




interactive(children=(Checkbox(value=False, description='50 Hz notch', indent=False), Checkbox(value=False, de…

<function __main__.temp_preprocess(apply_notch, apply_detrend, wt_denoise_level)>